<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/10_mfcc_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 — MFCC baseline'ları (Faz 1): EXP-010 ve EXP-011

**Bilimsel amaç** (D-012, D-031):
- **EXP-010 — sadık yeniden üretim:** Yayınlanmış çalışmanın tarifi ve değerlendirmesi olduğu gibi uygulanır:
  - orijinal `.m4a`, 22.05 kHz, kırpma;
  - 44 özellik;
  - görev başına tek 5-fold;
  - 14 model;
  - "en iyi"yi test fold'larına bakarak seçme.

  Soru: makaledeki AUC aralığına (≈0.65–0.77) ulaşıyor muyuz?
- **EXP-011 — bizim protokolümüz:** Aynı 44 özellik, ama farklı girdi ve protokolle:
  - girdi: harmonize önbellek (derin modellerle aynı girdi);
  - split'ler: ortak split dosyaları (5 tekrar × 5 fold);
  - modeller: önceden belirlenmiş LR (birincil), SVM-RBF ve gradient boosting; ayar iç fold'larda yapılır;
  - görev başına sonuçlar ve 7 görevin füzyonu;
  - aynı fold'larda referans çizgileri: yalnız yaş, yalnız bağlam.

  Derin modeller bununla karşılaştırılacak (RQ1).

**Üst sınır uyarısı (D-028):** Kayıt bağlamı tek başına AUC ≈ 0.93. Buradaki AUC'ler astım tespit başarısı değil, üst sınırdır.

**Çalışma ortamı:** GPU gerekmez: *Runtime → CPU*. Toplam ~2 saat:
- özellikler ~15 dk,
- EXP-010 ~35 dk,
- EXP-011 ~60 dk.

Bağlantı koparsa notebook'u baştan çalıştır. Biten görevler ve tekrarlar Drive'daki `partial/` klasöründen okunur ve atlanır.

## 0. Oturum kurulumu

In [2]:
import sys, platform, subprocess
print("Python:", sys.version.split()[0], "|", platform.platform())
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

Python: 3.13.16 | Linux-6.6.122+-x86_64-with-glibc2.39
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
RAW, DERIVED = DRIVE_ROOT / 'data_raw', DRIVE_ROOT / 'data_derived'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
AUDIO_ZIP   = RAW / 'soundData.zip'
AUDIO_DIR   = Path('/content/audio')
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
FEAT_DIR    = DERIVED / 'mfcc'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
for p in [AUDIO_ZIP, DERIVED / 'recording_map.csv', DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv',
          DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
FEAT_DIR.mkdir(exist_ok=True); EXPERIMENTS.mkdir(exist_ok=True)
print('Girdiler mevcut.')

Girdiler mevcut.


In [7]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', url, str(REPO_DIR)], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
assert r.returncode == 0, 'git başarısız'
COMMIT = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Repo commit:', COMMIT)

Cloning into '/content/asthma-voice-dl'...

Repo commit: 98428f4d0b25066abd4fe86cac1760439958ab9f


In [8]:
!pip install -q -r {REPO_DIR}/requirements-colab.txt
import librosa, imblearn, xgboost, catboost, sklearn
print('librosa', librosa.__version__, '| imblearn', imblearn.__version__, '| xgboost', xgboost.__version__,
      '| catboost', catboost.__version__, '| sklearn', sklearn.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.4 MB/s eta 0:00:00
librosa 0.11.0 | imblearn 0.14.2 | xgboost 3.4.1 | catboost 1.2.10 | sklearn 1.6.1


## 1. Hat testi (sahte veriyle, ~4 dk)
Gerçek veriye dokunmadan, üç şey doğrulanır:
- özellik çıkarımı iki modda da çalışıyor mu;
- yerleştirilmiş bir sinyal bulunuyor mu (AUC > 0.8);
- saf gürültüde AUC ≈ 0.5 mi (yani sızıntı yok mu).

Ayrıca referans çizgileri, füzyon ve kaldığı yerden devam test edilir. `Tüm kontroller geçti` görmeden devam etme.

In [10]:
!cd {REPO_DIR} && python tests/test_mfcc_pipeline.py

Tüm kontroller geçti (K1–K7).
  sinyal AUC 0.817 | gürültü AUC 0.517 | yaş referansı 0.859


## 2. Veriyi yerel diske hazırla (zip + önbellek; sha256 doğrulamalı)

In [11]:
import hashlib, shutil, zipfile, json, time
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

if not AUDIO_DIR.exists():
    shutil.copy(AUDIO_ZIP, '/content/soundData.zip')
    with zipfile.ZipFile('/content/soundData.zip') as z:
        z.extractall(AUDIO_DIR)
print('.m4a:', sum(1 for _ in AUDIO_DIR.rglob('*.m4a')))

info = json.loads((DRIVE_CACHE / 'cache_info.json').read_text())
LOCAL_CACHE.mkdir(exist_ok=True)
for name in ['audio_32k.f32', 'index.csv']:
    if not (LOCAL_CACHE / name).exists():
        shutil.copy(DRIVE_CACHE / name, LOCAL_CACHE / name)
    assert sha256(LOCAL_CACHE / name) == info['files_sha256'][name], f'{name}: sha256 farklı'
print('Önbellek yerel diskte ve doğrulandı.')

.m4a: 2407
Önbellek yerel diskte ve doğrulandı.


## 3. Özellikler (her biri bir kez üretilir; Drive'da varsa atlanır)
- `paper`: orijinal `.m4a` → 22.05 kHz → enerji tabanlı kırpma (EXP-010)
- `cache`: harmonize önbellek → 22.05 kHz (EXP-011)

In [12]:
F_PAPER, F_CACHE = FEAT_DIR / 'features_paper.csv', FEAT_DIR / 'features_cache.csv'
if not F_PAPER.exists():
    !cd {REPO_DIR} && python scripts/extract_mfcc_features.py --mode paper --audio-root {AUDIO_DIR} --recording-map {DERIVED}/recording_map.csv --out-csv {F_PAPER}
if not F_CACHE.exists():
    !cd {REPO_DIR} && python scripts/extract_mfcc_features.py --mode cache --cache-dir {LOCAL_CACHE} --out-csv {F_CACHE}
print('paper sha256:', sha256(F_PAPER)[:16], '| cache sha256:', sha256(F_CACHE)[:16])

paper sha256: b94da5c9712ea8f6 | cache sha256: f2ac9f55ccf9d579


## 4. EXP-010 — sadık yeniden üretim (~35 dk)
Görev başına sonuçlar Drive'da `experiments/EXP-010_mfcc-paper/partial/` altına yazılır. Kopma olursa bu hücreyi yeniden çalıştır: biten görevler atlanır.

In [14]:
!cd {REPO_DIR} && python scripts/run_mfcc_baselines.py --exp EXP-010 --features {F_PAPER} --context {DERIVED}/participant_context.csv --out-dir {EXPERIMENTS}/EXP-010_mfcc-paper --report-dir {REPO_DIR}/reports/mfcc --registry {REPO_DIR}/results/registry.csv

EXP-010: 342 katılımcı (283 astım / 59 sağlıklı), 2393 kayıt
  görev aaa: önceki sonuç kullanıldı (EXP-010_1.csv)
  görev araba: önceki sonuç kullanıldı (EXP-010_2.csv)
  görev ana: önceki sonuç kullanıldı (EXP-010_3.csv)
  görev ordu: önceki sonuç kullanıldı (EXP-010_4.csv)
  görev gelecek: önceki sonuç kullanıldı (EXP-010_5.csv)
  görev titiz: önceki sonuç kullanıldı (EXP-010_6.csv)
  görev ünlem: önceki sonuç kullanıldı (EXP-010_7.csv)
# EXP-010 — MFCC baseline (otomatik üretildi; katılımcı ID'si içermez)

**Üst sınır uyarısı (D-028):** Kayıt bağlamı (tarih + saat) sesi kullanmadan etiketi AUC ≈ 0.93 ile ele veriyor; aşağıdaki AUC'ler astım tespit başarısı değil, bağlamdan etkilenmiş olabilecek üst sınırlardır. Bağlam değerlendirmesi model geliştirme bitince yapılacak (D-023).

- Katılımcı: 342; özellik dosyası sha256 `b94da5c9712ea8f6…`; süre 355.7 s
- Sürümler: {'python': '3.13.16', 'scikit-learn': '1.6.1', 'git_commit': '98428f4d0b25066abd4fe86cac1760439958ab9f', 'imblearn': '0.1

## 5. EXP-011 — bizim protokolümüz (~60 dk)
Tekrar başına sonuçlar `experiments/EXP-011_mfcc-protocol/partial/` altına yazılır; biten tekrarlar atlanır.

In [15]:
!cd {REPO_DIR} && python scripts/run_mfcc_baselines.py --exp EXP-011 --features {F_CACHE} --context {DERIVED}/participant_context.csv --splits-dir {DERIVED}/splits --out-dir {EXPERIMENTS}/EXP-011_mfcc-protocol --report-dir {REPO_DIR}/reports/mfcc --registry {REPO_DIR}/results/registry.csv

EXP-011: 342 katılımcı (283 astım / 59 sağlıklı), 2393 kayıt
  tekrar 0: önceki sonuç kullanıldı
  tekrar 1: önceki sonuç kullanıldı
  tekrar 2: önceki sonuç kullanıldı
  tekrar 3: önceki sonuç kullanıldı
  tekrar 4: önceki sonuç kullanıldı
# EXP-011 — MFCC baseline (otomatik üretildi; katılımcı ID'si içermez)

**Üst sınır uyarısı (D-028):** Kayıt bağlamı (tarih + saat) sesi kullanmadan etiketi AUC ≈ 0.93 ile ele veriyor; aşağıdaki AUC'ler astım tespit başarısı değil, bağlamdan etkilenmiş olabilecek üst sınırlardır. Bağlam değerlendirmesi model geliştirme bitince yapılacak (D-023).

- Katılımcı: 342; özellik dosyası sha256 `f2ac9f55ccf9d579…`; süre 109.8 s
- Sürümler: {'python': '3.13.16', 'scikit-learn': '1.6.1', 'git_commit': '98428f4d0b25066abd4fe86cac1760439958ab9f', 'imblearn': '0.14.2', 'xgboost': '3.4.1', 'catboost': '1.2.10', 'librosa': '0.11.0'}

## Bütün modeller

| görev | model | fold AUC ort ± SD (fold %2.5–97.5) | havuzlanmış OOF AUC [%95 CI] | dengeli doğr. | duyarlılık 

## 6. Raporları GitHub'a gönder (ayrı dal)
Yalnız iki şey commit edilir:
- `reports/mfcc/` (agrega);
- `results/registry.csv`.

Katılımcı düzeyindeki tahminler Drive'da kalır (D-014).

In [16]:
BRANCH = 'mfcc-baseline-results'
def git(*a):
    r = subprocess.run(['git', '-C', str(REPO_DIR), *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***').strip())
    return r
git('config', 'user.name', 'alperarslan19')
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
git('checkout', '-B', BRANCH)
git('add', 'reports/mfcc', 'results/registry.csv')
staged = git('diff', '--cached', '--name-only').stdout.split()
assert staged and all(s.startswith('reports/mfcc/') or s == 'results/registry.csv' for s in staged), f'Beklenmeyen dosya: {staged}'
assert git('commit', '-m', 'EXP-010/011: MFCC baseline sonuçları (Colab)').returncode == 0
assert git('push', f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git', BRANCH).returncode == 0
print(f'PR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')



M	results/registry.csv
Reset branch 'mfcc-baseline-results'

reports/mfcc/EXP-010_mfcc.json
reports/mfcc/EXP-010_mfcc.md
reports/mfcc/EXP-011_mfcc.json
reports/mfcc/EXP-011_mfcc.md
results/registry.csv
[mfcc-baseline-results f1da53a] EXP-010/011: MFCC baseline sonuçları (Colab)
 5 files changed, 2913 insertions(+), 40 deletions(-)
 create mode 100644 reports/mfcc/EXP-010_mfcc.json
 create mode 100644 reports/mfcc/EXP-010_mfcc.md
 create mode 100644 reports/mfcc/EXP-011_mfcc.json
 create mode 100644 reports/mfcc/EXP-011_mfcc.md
remote: 
remote: Create a pull request for 'mfcc-baseline-results' on GitHub by visiting:        
remote:      https://github.com/alperarslan19/asthma-voice-dl/pull/new/mfcc-baseline-results        
remote: 
To https://github.com/alperarslan19/asthma-voice-dl.git
 * [new branch]      mfcc-baseline-results -> mfcc-baseline-results
PR aç: https://github.com/alperarslan19/asthma-voice-dl/pull/new/mfcc-baseline-results
